In [ ]:
!pip install -q pandas
!pip install -q matplotlib
!pip install -q seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data = pd.read_csv("../data/data_part1.csv")

In [ ]:
def get_quantplot(series):
    quantiles = np.arange(0, 1.00, 0.01)
    q_values = series.quantile(quantiles)
    
    plt.figure(figsize=(10, 6))
    q_values.plot(kind='line', linewidth=2, marker='o', markersize=3)
    plt.title('График возрастания квантилей цены')
    plt.xlabel('Квантиль (0-1)')
    plt.ylabel('Значение')
    plt.grid(True, alpha=0.3)
    plt.show()


In [ ]:
data.head()

In [ ]:
data.isna().sum()

In [ ]:
data = data.dropna(subset=["Цена"])

In [ ]:
data.columns

In [ ]:
data = data.drop([
    'Телефон', 'Оператор', 'Контактное лицо (автор объявления)',
    'Персона для контактов', 'Регион мобильного телефона', 'Номер подменён',
    'Тип объявления', 'Категория1', 'Категория2'
], axis = 1)

In [ ]:
data.columns

In [ ]:
data = data.set_index(['ID на сайте', 'Источник',])

In [ ]:
data.head()

In [ ]:
data.loc[(320133256, 'cian.ru')]

In [ ]:
data["Описание"] = "НАЗВАНИЕ: " + data["Название"] + "\n\nОПИСАНИЕ: " + data["Описание"]

In [ ]:
data.loc[(320133256, "cian.ru"), ("Название", "Описание")]

In [ ]:
data

In [ ]:
data = data.drop("Название", axis=1)

In [ ]:
data["Доп.параметры"].iloc[10]

In [ ]:
def parse_parameners(v):
    return dict(pair.split("=") for pair in v.split('|'))
parse_parameners(data["Доп.параметры"].iloc[10])

In [ ]:
_ = data["Доп.параметры"].apply(parse_parameners)
params_parsed = pd.json_normalize(_)
params_parsed.index = data.index

In [ ]:
params_parsed.isna().sum() / params_parsed.shape[0] * 100

In [ ]:
params_parsed.nunique()

In [ ]:
params_parsed = params_parsed.drop(["Тип объявления"], axis=1)

In [ ]:
data = pd.concat(
    [
        data.drop(["Доп.параметры"], axis=1),
        params_parsed
    ],
    axis=1
)

In [ ]:
data.dtypes

In [ ]:
data["Дата"] = pd.to_datetime(data["Дата"])
data["Дата"]

In [ ]:
data["Расстояние до метро, км"] = data["Расстояние до метро, км"].replace({"неизвестно": np.nan})
data [[
    "Расстояние до метро, км",
    "Этаж",
    "Этажность здания",
    "Общая площадь", "Цена"
]] = data [[
    "Расстояние до метро, км",
    "Этаж",
    "Этажность здания",
    "Общая площадь", "Цена"
]].astype(float)

In [ ]:
data.isna().sum()

In [ ]:
data[["Общая площадь", "Ед. измерения площади"]].dropna()

In [ ]:
data.loc[data["Ед. измерения площади"] == "сот.",
    "Общая площадь"] *= 100
data.loc[data["Ед. измерения площади"] == "га.",
    "Общая площадь"] *= 10000

data.loc[data["Ед. измерения площади"] == "га.",
    "Общая площадь"]

In [ ]:
data = data.drop("Ед. измерения площади", axis=1)

In [ ]:
data

# -------------------------------------------------------------------------------


In [ ]:
data = data[data["Вид объекта"] != "Коммерческая земля"].copy()
data = data[data["Вид объекта"] != "Складское помещение"].copy()
data = data[data["Вид объекта"] != "Производственное помещение"].copy()
#data = data[data["Вид объекта"] != "Офисное помещение"].copy()
data = data[data["Вид объекта"] != "Гостиница"].copy()
#data = data[data["Вид объекта"] != "Здание"].copy()
data = data[data["Общая площадь"] <= 2000].copy()
data = data[data["Общая площадь"] >= 20].copy()

In [ ]:
data2 = data.copy()
mask = (data2["Цена"] >= data2["Цена"].quantile(0.97))
data2.loc[mask, "Цена"] /= 12.0
data = data2.copy()

In [ ]:
data["Вид объекта"].value_counts()

In [ ]:
data["Общая площадь"].describe()

In [ ]:
data

In [ ]:
plt.hist(data["Общая площадь"], bins=30)
plt.show()

In [ ]:
data = data[data["Цена"] <= 2000000].copy()
data = data[data["Цена"] >= 20000].copy()

In [ ]:
plt.hist(data["Цена"], bins=30)
plt.show()

In [ ]:
plt.hist(np.log1p(data["Цена"]), bins = 30)
plt.show()

In [ ]:
sns.histplot(data, x="Цена", bins=50)
plt.show()

In [ ]:
sns.histplot(data, x="Цена",hue="Вид объекта", bins=50)
plt.xlim([0, 2e6])
plt.show()

In [ ]:
data["Log(Цена)"] = np.log1p(data["Цена"])
sns.histplot(data, x="Log(Цена)",hue="Вид объекта", bins=50)
plt.show()

In [ ]:
dfg = data.groupby(by="Вид объекта")["Цена"].mean()
dfg.astype(int)

In [ ]:
sns.barplot(dfg)
plt.xticks(rotation=80)
plt.show()

In [ ]:
sns.histplot(data, x="Цена",hue="Вид объекта", bins=50)
plt.show()

In [ ]:
sns.histplot?

In [ ]:
sns.histplot(data, x="Общая площадь",hue="Вид объекта", bins=50, stat="density", common_norm=False, kde=True)
plt.show()

In [ ]:
sns.histplot(data, x="Цена",hue="Вид объекта", bins=50, stat="density", common_norm=False, kde=True)
plt.show()

In [ ]:
data["Цена / м2"] = data["Цена"] / data["Общая площадь"]
data["Цена / м2"].describe([0.01, 0.05, 0.1, 0.2, 0.5, 0.8, 0.9, 0.95, 0.99])

In [ ]:
data = data[data["Цена / м2"] <= 10000].copy()
data = data[data["Цена / м2"] >= 300].copy()

In [ ]:
data.columns

In [ ]:
data = data.drop(["Регион"], axis=1)

In [ ]:
data1 = data.dropna(subset=["Расстояние до метро, км"])
sns.regplot(
    y='Цена / м2', 
    x='Расстояние до метро, км', 
    data=data1)
plt.show()

In [ ]:
data["Тип автора"].unique()

In [ ]:
sns.histplot(data, x="Цена / м2",hue="Вид объекта", bins=50, stat="density", common_norm=False, kde=True)
plt.show()

In [ ]:
dfg = data.groupby(by="Вид объекта")["Цена / м2"].mean()
dfg.astype(int)
print(dfg)
sns.barplot(dfg)
plt.xticks(rotation=80)
plt.show()

In [ ]:
data

In [ ]:
data["Город"].unique()

In [ ]:
data = data.drop("Категория", axis=1)

In [ ]:
data

In [ ]:
dfg = data.groupby(by="Этажность здания")["Цена / м2"].mean()
dfg.astype(int)
print(dfg)
sns.barplot(dfg)
plt.xticks(rotation=80)
plt.show()

In [ ]:
dfg = data.groupby(by="Этаж")["Цена / м2"].mean()
dfg.astype(int)
print(dfg)
sns.barplot(dfg)
plt.xticks(rotation=80)
plt.show()

In [ ]:
data = data.drop("Log(Цена)",  axis=1)

In [ ]:
data["Вид объекта"] = data["Вид объекта"].replace({'Офисное помещение': 'Офис / Здание', 'Здание': 'Офис / Здание',
                                    'Торговое помещение': 'Торговое / свободного вида',
                                    'Помещение свободного назначения': 'Торговое / свободного вида'})
data["Вид объекта"].unique()

In [ ]:
data["Тип автора"] = data["Тип автора"].replace({'Частное лицо (фильтр)': 'Частное лицо'})
data["Тип автора"].unique()

In [ ]:
data["Москва?"] = (data["Город"] == "Москва").astype(int)
data["Город"].sum()

In [ ]:
data.to_csv("../data/data_transformed_part1.csv")

In [ ]:
data